# Method 2 — Bi-Encoder (Kaggle T4)

Phase 2 của `docs/method2_plan.md`: train 2 vòng, pre-compute index, hiệu chỉnh ngưỡng. Ngân sách ~4h GPU.

**Ba quy tắc sống còn trên Kaggle** (§8 `docs/method2_plan.md`):

1. Bật **Save & Run All (Commit)** cho job dài — session tương tác bị ngắt sau ~20 phút không tương tác, commit run chạy nền đủ 12h.
2. Checkpoint mỗi 500 step vào `/kaggle/working`, và **luôn** hỗ trợ `resume_from`.
3. Cache model HuggingFace thành Kaggle Dataset (`BAAI/bge-m3` ~2.3GB) thay vì tải lại mỗi session.


In [ ]:
# ===== Cell 0: dò dataset + HF cache =====
# PHẢI chạy trước mọi import transformers: thư viện chốt cache lúc import,
# set HF_HOME sau đó thì không còn tác dụng.
import os
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')


def _dirs_within(base: Path, max_depth: int = 4):
    """Mọi thư mục tới độ sâu `max_depth`, bỏ qua `hub/` cho nhanh."""
    frontier, seen = [base], []
    for _ in range(max_depth):
        nxt = []
        for d in frontier:
            try:
                children = [c for c in d.iterdir() if c.is_dir() and c.name != 'hub']
            except (PermissionError, OSError):
                continue
            seen.extend(children)
            nxt.extend(children)
        frontier = nxt
    return seen


def find_root(marker: str, label: str) -> Path:
    """Tìm thư mục chứa `marker`.

    Kaggle mount theo dạng /kaggle/input/datasets/<user>/<ds>/<ds>/, và số tầng
    đổi theo cách upload. Dò theo marker thì không phải hardcode username hay
    độ sâu — upload kiểu nào cũng tìm ra.
    """
    for d in [INPUT_ROOT] + _dirs_within(INPUT_ROOT):
        if (d / marker).exists():
            return d
    raise SystemExit(
        f'Không tìm thấy {label}: không thư mục nào dưới {INPUT_ROOT} có {marker}.\n'
        'Kiểm tra đã Add đủ 3 dataset ở sidebar Input chưa.'
    )


SRC_ROOT = find_root('src/models/preflight.py', 'dataset src')
DATA_ROOT = find_root('method2/manifest.json', 'dataset data')
HF_HOME = find_root('hub/models--BAAI--bge-m3', 'dataset hf-cache')

print('SRC :', SRC_ROOT)
print('DATA:', DATA_ROOT)
print('HF  :', HF_HOME)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

# Có thư mục model chưa đủ — thiếu file trọng số thì lỗi chỉ lộ ra lúc nạp
# model, sau khi đã tốn thời gian cài đặt và copy.
for name in ('models--BAAI--bge-m3', 'models--xlm-roberta-base'):
    weights = [
        f for f in (HF_HOME / 'hub' / name).rglob('*')
        if f.is_file() and f.suffix in ('.safetensors', '.bin') and f.stat().st_size > 10**8
    ]
    assert weights, f'{name}: không có file trọng số > 100 MB'
    print(f'  {name}: {max(f.stat().st_size for f in weights) / 1024**3:.2f} GB')
print('\nHF cache OK')


In [ ]:
# ===== Cell 1: env — PIN version =====
# Ba package này quyết định API training VÀ tên metric của
# InformationRetrievalEvaluator. Đổi bản là đổi khoá metric, hỏng cả
# load_best_model_at_end lẫn khả năng so sánh giữa các run.
!pip install -q 'transformers==5.15.1' 'sentence-transformers==6.0.0' 'peft==0.20.0' \
                accelerate jsonschema rank_bm25 datasets

# PEFT 0.20 raise nếu image có torchao < 0.16. Method 2 không dùng
# torchao quantization nên gỡ hẳn là xong.
!pip uninstall -y -q torchao 2>/dev/null || true

# torch KHÔNG pin: Kaggle cài sẵn bản CUDA riêng, ép cài lại vừa chậm vừa
# dễ lệch CUDA runtime của image. Chỉ ghi nhận version vào manifest.
import torch

free, total = torch.cuda.mem_get_info()
n_gpu = torch.cuda.device_count()
print(torch.cuda.get_device_name(0), f'{free/1024**3:.1f} / {total/1024**3:.1f} GB free')
print('số GPU:', n_gpu)

# sentence-transformers tự bọc DataParallel khi thấy >1 GPU. Với GradCache
# gọi model hàng trăm lần mỗi step thì phí đồng bộ cộng dồn rất nhanh.
if n_gpu > 1:
    print('  >1 GPU — truyền --single-gpu cho MỌI lệnh train')

# T4 là Turing (sm_75), KHÔNG có bf16 phần cứng. torch vẫn có thể báo
# is_bf16_supported()=True vì hỗ trợ qua emulation, chậm hơn fp16.
# Giữ fp16 bất kể giá trị này.
print('bf16 (emulated trên T4, vẫn dùng fp16):', torch.cuda.is_bf16_supported())


In [ ]:
# ===== Cell 2: copy code + data vào /kaggle/working =====
# Dataset chỉ đọc, mà code ghi checkpoint và dùng đường dẫn tương đối, nên
# phải copy sang thư mục ghi được. Dùng path đã dò ở Cell 0.
import shutil

WORK = Path('/kaggle/working')
# `scripts` cần thiết: benchmark_biencoder.py chạy trên Kaggle.
for name in ('src', 'configs', 'scripts'):
    target = WORK / name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(SRC_ROOT / name, target)

# Dataset data bắt đầu thẳng bằng method2/ custom_vi/ benchmark_vi/ (KHÔNG có
# tầng `data/`), còn code tham chiếu `data/method2/...` → copy vào data/.
data_dir = WORK / 'data'
if data_dir.exists():
    shutil.rmtree(data_dir)
data_dir.mkdir(parents=True)
for child in DATA_ROOT.iterdir():
    dest = data_dir / child.name
    shutil.copytree(child, dest) if child.is_dir() else shutil.copy2(child, dest)

%cd /kaggle/working

import json, glob, sys
sys.path.insert(0, '/kaggle/working')
# HF_HOME đã set ở Cell 0, kế thừa sang mọi tiến trình con `!python`.

print('src    :', sorted(p.name for p in (WORK / 'src').iterdir()))
print('data   :', sorted(p.name for p in data_dir.iterdir()))


In [ ]:
# ===== Cell 3: kiểm tra bản copy TRƯỚC khi preflight =====
# Preflight kiểm tra tính đúng đắn của dữ liệu; cell này kiểm tra bước copy —
# tách ra để khi hỏng thì biết ngay là hỏng ở đâu.
REQUIRED = [
    'data/method2/decontamination.json',
    'data/method2/manifest.json',
    'data/method2/tool_pool.json',
    'data/method2/biencoder/train.jsonl',
    'data/method2/biencoder/val.jsonl',
    'data/method2/biencoder/pairs_stats.json',
    'data/method2/crossencoder/train.jsonl',
    'data/method2/crossencoder/val.jsonl',
    'data/method2/label_stats.json',
    'data/custom_vi/v1/test_seen.jsonl',
    'data/benchmark_vi/test.jsonl',
    'configs/method2/biencoder.yaml',
    'configs/method2/pinned_versions.json',
    'src/models/preflight.py',
]
missing = []
for rel in REQUIRED:
    path = WORK / rel
    if path.exists() and path.stat().st_size > 0:
        print(f'  {path.stat().st_size / 1024**2:8.2f} MB  {rel}')
    else:
        missing.append(rel)
        print(f'  {"THIẾU":>11}  {rel}')
assert not missing, f'Copy chưa đủ: {missing}'

# import được thì mới chắc src/ copy nguyên vẹn.
import importlib

importlib.import_module('src.models.preflight')
manifest = json.load(open('data/method2/manifest.json', encoding='utf-8'))
print('\nsnapshot commit:', manifest.get('git_commit'))
print('copy OK')


## Pre-flight — cổng fail-closed TRƯỚC mọi training

```
decontamination.json tồn tại
        ↓
SHA-256 == manifest.json
        ↓
overlap train/val/test == 0
        ↓
unseen positive leakage == 0
        ↓
package versions khớp bản đã pin
        ↓
CHO PHÉP TRAIN
```

Thiếu file hoặc hash lệch → job dừng ngay, **không rebuild tự động**. Nếu
experiment chính tự dựng lại index từ dữ liệu đang có trên máy thì ta mất
đúng thứ cần đảm bảo: bằng chứng model được train trên đúng split đã kiểm
định. Rebuild là lệnh preprocessing riêng, chạy ở local rồi upload lại:
`python -m src.models.sources decontaminate && python -m src.models.sources manifest`

Vì sao `val ∩ test` là rủi ro nặng nhất: dù không train trên query đó, việc
chọn checkpoint/hyperparameter bằng val vẫn khiến metric test lạc quan hơn
thực tế. `data/benchmark_vi` **giữ nguyên** — decontamination nằm ở tầng
dataset của Method 2 nên bốn method vẫn được đánh giá trên cùng một tập test.


In [ ]:
# Exit code != 0 → dừng notebook, không chạy tiếp cell training nào.
!python -m src.models.preflight \
    --config configs/method2/biencoder.yaml \
    --require-gpu T4 \
    --output results/method2/preflight.json

preflight = json.load(open('results/method2/preflight.json', encoding='utf-8'))
assert preflight['passed'], f"Preflight KHÔNG ĐẠT: {preflight['failures']}"
print('preflight PASS —', len(preflight['checks']), 'check')


In [ ]:
# Số liệu split để đối chiếu bằng mắt trước khi tiêu giờ GPU.
stats = json.load(open('data/method2/biencoder/pairs_stats.json', encoding='utf-8'))
decon = stats['decontamination']

print('unique query/split :', stats['unique_queries_per_split'])
print('positive pairs     :', stats['n_positive_pairs'])
print('negative samples   :', stats['n_negative_samples'])
print('query trùng split  :', decon['n_overlapping_queries'], decon['overlapping_queries'])
print('sample bị loại     :', decon['rows_dropped_total'], decon['rows_dropped_by_transition'])
print('overlap còn lại    :', stats['split_overlap_after'])


# Run 1a — Benchmark cấu hình (BẮT BUỘC trước smoke)

Lần chạy đầu trên T4 cho **475 s/step**: 100 step mất 13.2 giờ, một epoch
mất 49 giờ, trong khi plan dự toán 50-70 phút/epoch. Lệch ~45× nên phải tìm
cấu hình dùng được trước, đừng chạy tiếp smoke 100 step.

Vì sao mỗi step đắt: `CachedMNRL` không phải một forward/backward bình
thường. Effective batch 256, mỗi sample có anchor + positive + 4 negative →
**1,536 lượt encode**. Chia mini_batch 8 thành 192 chunk, GradCache chạy
**hai** pha (forward no-grad để cache, rồi forward+backward tính lại) →
~384 lần gọi model mỗi step. Mỗi lần chỉ 8×192 = 1,536 token, quá nhỏ để lấp
đầy T4 nên phần lớn thời gian là overhead — cộng thêm DataParallel giữa 2 GPU
thì nhân lên tiếp.

| Case | GPU | batch | mini | ckpt | đổi gì so với case trước |
|---|---|---|---|---|---|
| A | 1×T4 | 256 | 8 | on | tách ảnh hưởng DataParallel |
| B | 1×T4 | 256 | 16 | on | nửa số lần gọi model |
| C | 1×T4 | 256 | 32 | on | 1/4 số lần gọi model |
| D | 1×T4 | 128 | 32 | on | giảm effective batch |
| E | 1×T4 | 256 | 32 | off | tắt grad checkpointing |

A→C chỉ đổi **tốc độ**. D đổi **chất lượng**: MNRL mạnh lên theo số in-batch
negative, giảm batch là giảm negative — chỉ dùng khi A–C không đủ, và phải
ghi rõ vào báo cáo.


In [ ]:
# 5 step mỗi case, tắt eval (eval trên corpus 4,4k tool làm nhiễu số đo).
# `sec_per_step` lấy từ `train_runtime` của HF nên KHÔNG gồm thời gian nạp
# BGE-M3 — với run 5 step thì nạp model lấn át hoàn toàn wall-clock.
!python scripts/method2/benchmark_biencoder.py \
    --steps 5 \
    --config configs/method2/biencoder.yaml \
    --output results/method2/benchmark_biencoder.json


### Chốt cấu hình

Chọn case nhanh nhất mà VRAM còn an toàn (< ~13 GB để chừa chỗ cho eval),
rồi ghi vào `configs/method2/biencoder.yaml` trước khi chạy smoke.

Ngưỡng thực dụng: **> 60 s/step là chưa dùng được** — 370 step/epoch × 3
epoch mà 60 s/step đã là 18 giờ, vượt quota tuần.


In [ ]:
bench = json.load(open('results/method2/benchmark_biencoder.json', encoding='utf-8'))
ok = [b for b in bench if b['ok'] and b['hours_per_epoch']]
assert ok, 'Không case nào chạy được — xem log ở trên'

# Chọn theo GIỜ/EPOCH, KHÔNG theo s/step: giảm effective batch làm s/step
# đẹp hẳn lên nhưng số step mỗi epoch tăng đúng bấy nhiêu lần.
best = min(ok, key=lambda b: b['hours_per_epoch'])
print('nhanh nhất theo epoch:', best['case']['name'],
      f"{best['hours_per_epoch']:.2f} h/epoch,",
      f"{best['sec_per_step']:.1f} s/step × {best['steps_per_epoch']:,} step,",
      f"peak {best['peak_vram_mb']:.0f} MB")
print(f"3 epoch ≈ {best['hours_3_epochs']:.1f} h")

# Ghi cấu hình thắng cuộc vào YAML để smoke và full train dùng chung.
import yaml

cfg_path = 'configs/method2/biencoder.yaml'
cfg = yaml.safe_load(open(cfg_path, encoding='utf-8'))
cfg['train']['batch_size'] = best['case']['batch_size']
cfg['train']['mini_batch_size'] = best['case']['mini_batch_size']
cfg['train']['gradient_checkpointing'] = best['case']['grad_checkpointing']
yaml.safe_dump(cfg, open(cfg_path, 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
print('đã ghi vào', cfg_path)

if best['hours_3_epochs'] > 6:
    print('\nVƯỢT NGÂN SÁCH Phase 2 (~4h):',
          f"{best['hours_3_epochs']:.1f} h cho 3 epoch")
    print('Thử tiếp: mini_batch 64, n_hard_negatives 4→2 (cắt ~33% compute),')
    print('hoặc giảm số epoch, hoặc max_seq_length 192 → 128.')

if best['peak_vram_mb'] > 9000:
    print('\nVRAM', f"{best['peak_vram_mb']:.0f} MB đo khi TẮT eval.",
          'Chạy lại case này CÓ eval trước khi tin là an toàn —')
    print('evaluator encode ~4,4k tool doc trong khi trạng thái train vẫn giữ VRAM.')
if best['case']['batch_size'] != 256:
    print('\nLƯU Ý: effective batch giảm còn', best['case']['batch_size'],
          '— đổi CHẤT LƯỢNG chứ không chỉ tốc độ, phải ghi vào báo cáo.')


# Full training

Cấu hình đã chốt trong `configs/method2/biencoder.yaml` từ benchmark:
batch 256 · mini_batch 32 · grad checkpointing **tắt** · **2 epoch** ·
`n_hard_negatives` **2**. Ước tính **~4.1 h**.

Ba thứ đã cắt so với plan gốc, để vừa quota — phải ghi vào báo cáo:

| Cắt gì | Từ → đến | Tiết kiệm | Ảnh hưởng |
|---|---|---|---|
| epoch | 3 → 2 | 3.1 h | ít, MNRL hội tụ nhanh |
| `n_hard_negatives` | 4 → 2 | 3.1 h | **đổi chất lượng** — ablation §6.4 |
| Round 2 (mining) | có → không | 4.1 h | không có hard negative đã mine |

Smoke đã chạy trước đó và pass phần chức năng (resume đúng mốc 100, loss
4.61 → 2.93 không NaN, checkpoint đọc lại được) nên bỏ khỏi luồng này.

OOM thì đặt `gradient_checkpointing: true` rồi chạy lại cell — nó tự resume
từ checkpoint gần nhất (save mỗi 100 step).


## Round 1 — train với hard negative round-0

`CachedMultipleNegativesRankingLoss` (GradCache) cho effective batch 256 với
mini_batch 8. Gradient accumulation **không** thay thế được: nó chỉ chia nhỏ
update chứ không làm tăng số in-batch negative.


In [ ]:
import re

RUN = '/kaggle/working/artifacts/method2/biencoder/run01'

# sorted() theo tên là sai: 'checkpoint-1000' < 'checkpoint-500' theo thứ tự
# chữ, nên sẽ resume nhầm checkpoint cũ hơn.
ckpts = sorted(
    glob.glob(f'{RUN}/checkpoint-*'),
    key=lambda p: int(re.search(r'(\d+)$', p).group(1)),
)
resume = ckpts[-1] if ckpts else None
print('resume from:', resume or '(chưa có checkpoint — train từ đầu)')

# KHÔNG truyền cờ khi không có checkpoint: `{None}` nội suy thành chuỗi
# "None", HF Trainer coi đó là đường dẫn rồi đi tải từ Hub và chết vì offline.
resume_arg = f'--resume-from {resume}' if resume else ''
# --single-gpu: benchmark đã đo ở chế độ 1 GPU, để DataParallel bật lại thì
# tốc độ thực tế khác hẳn con số đã chốt.
!python -m src.models.biencoder.train train \
    --config configs/method2/biencoder.yaml \
    --output-dir {RUN} \
    --single-gpu {resume_arg}


## Round 2 — mine hard negatives rồi train lại **từ base** (TUỲ CHỌN)

Lấy tool sai nhưng xếp hạng cao (bỏ top-1 để tránh false negative). Round 2
train lại từ checkpoint gốc, không train tiếp từ round 1.

**Mặc định TẮT** (`mining.enabled: false`) vì đây là một lần train đầy đủ
nữa, tốn bằng Round 1. Bật lại khi còn quota; nếu bỏ thì phải ghi rõ trong
báo cáo là kết quả Bi-Encoder chỉ có Round 1, chưa mine hard negative.


In [ ]:
import subprocess, sys

import yaml

_cfg = yaml.safe_load(open('configs/method2/biencoder.yaml', encoding='utf-8'))
RUN_ROUND2 = bool(_cfg.get('mining', {}).get('enabled', False))
RUN2 = '/kaggle/working/artifacts/method2/biencoder/run02'

# subprocess thay vì `!` trong `if`: exit code hiện ra rõ ràng, và một lệnh
# hỏng không bị trôi qua trong Save & Run All.
if RUN_ROUND2:
    subprocess.run(
        [sys.executable, '-m', 'src.models.biencoder.train', 'mine',
         '--config', 'configs/method2/biencoder.yaml', '--model', f'{RUN}/final'],
        check=True,
    )
    cfg_text = open('configs/method2/biencoder.yaml', encoding='utf-8').read()
    open('configs/method2/biencoder.yaml', 'w', encoding='utf-8').write(
        cfg_text.replace('biencoder/train.jsonl', 'biencoder/train_mined.jsonl')
    )
    subprocess.run(
        [sys.executable, '-m', 'src.models.biencoder.train', 'train',
         '--config', 'configs/method2/biencoder.yaml',
         '--output-dir', RUN2, '--single-gpu'],
        check=True,
    )
else:
    print('Round 2 TẮT (mining.enabled=false) — dùng checkpoint của Round 1.')

# Mọi bước sau dùng FINAL, không trỏ cứng vào RUN2: bỏ Round 2 thì RUN2
# không tồn tại và index/calibrate sẽ chết vì không tìm thấy model.
FINAL = RUN2 if RUN_ROUND2 else RUN
print('checkpoint dùng cho các bước sau:', FINAL)


## Pre-compute index + hiệu chỉnh ngưỡng trên **val**


In [ ]:
!python -m src.models.biencoder.index \
    --config configs/method2/biencoder.yaml --model {FINAL}/final

# τ và τ_call CHỈ được hiệu chỉnh trên val, rồi freeze trước khi chạy test.
!python -m src.models.biencoder.evaluate calibrate \
    --config configs/method2/biencoder.yaml \
    --model {FINAL}/final \
    --pairs data/method2/biencoder/val.jsonl \
    --output {FINAL}/thresholds.json


## Gate để qua Phase 3

| Metric | Tập | Ngưỡng |
|---|---|---|
| Recall@1 | custom `val_seen` | ≥ 0.90 |
| Recall@1 | custom `val_unseen` | ≥ 0.75 |
| Recall@5 | custom `val_unseen` | ≥ 0.92 |
| Negative Recall @ τ | custom val negative | ≥ 0.80 |

Không đạt → thử theo thứ tự: (a) thêm param name vào document text,
(b) tăng hard negative lên 8, (c) đổi sang `AITeamVN/Vietnamese_Embedding`,
(d) full fine-tune thay LoRA.


In [ ]:
!python -m src.models.biencoder.evaluate evaluate \
    --config configs/method2/biencoder.yaml \
    --model {FINAL}/final \
    --pairs data/method2/biencoder/val.jsonl \
    --output results/method2/metrics/biencoder_val.json

report = json.load(open('results/method2/metrics/biencoder_val.json', encoding='utf-8'))
for slice_name, metrics in report['by_source_key'].items():
    print(slice_name, {k: v for k, v in metrics.items() if 'recall@' in k or k == 'mrr'})


## Run manifest — chốt lại toàn bộ mục audit

Train xong mà không audit được thì coi như chưa train. Cell này gom: commit
SHA (kèm cờ dirty), config YAML thực tế, fingerprint dataset + tool pool,
query counts và overlap theo split, số positive/negative pair, checkpoint,
best step + metric đã dùng để chọn, VRAM peak, thời lượng train, và
Recall@1/@5/@10 + MRR. Thiếu mục nào thì `audit_complete.missing` liệt kê ra.

`checkpoint_selection.available_metrics` cho biết tên metric thật của
`InformationRetrievalEvaluator` ở phiên bản đang chạy — khai vào
`train.metric_for_best_model` cho lần chạy sau.


In [ ]:
!python -m src.models.run_manifest \
    --run-dir {FINAL} \
    --config configs/method2/biencoder.yaml \
    --stage biencoder \
    --report retrieval=results/method2/metrics/biencoder_val.json

manifest = json.load(open(f'{FINAL}/run_manifest.json', encoding='utf-8'))
missing = manifest['audit_complete']['missing']
print('thiếu:', missing or 'không thiếu mục nào')
print('Recall/MRR:', manifest.get('retrieval_gate', {}).get('metrics'))
print('VRAM peak MB:', manifest['train']['peak_vram_mb'])
print('thời lượng (giờ):', manifest['train']['duration_hours'])
print('chọn checkpoint:', manifest['train']['checkpoint_selection'])
assert not missing, f'Chưa đủ artifact để audit: {missing}'


In [ ]:
# ===== Lưu artifact =====
# Kaggle chỉ giữ /kaggle/working (20GB). Nén để tải về hoặc làm Dataset mới.
!tar czf /kaggle/working/biencoder_run.tar.gz -C /kaggle/working/artifacts/method2 .
!du -h /kaggle/working/biencoder_run.tar.gz
